# Evaluation of the fine-tuned Qwen2.5-Coder-1.5B (Run 1) with the SAME benchmark code as the base model
Cells are copied from `benchmark_qwen` and `complement_gpu` (generation, prompts, HumanEval, energy, security prompts, pass@k). Only the model-loading cell is new.

**Two sessions, one `VARIANT` each** (set it in cell 4):
- `ft_bf16`: bf16 base + LoRA adapter merged into the weights (needs the adapter zip added as a Kaggle Input)
- `base`: control run of the plain base model in THIS environment (library versions differ from the original baseline runs)

GPU: Tesla T4, one GPU, Internet on. Download the generated zip at the end.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
REPO_URL  = "https://github.com/FatiAllali123/SLM-Software-Engineering.git"
REPO_NAME = "SLM-Software-Engineering"
BRANCH    = "main"

if not os.path.exists("../config.json"):
    !git clone --depth 1 -b {BRANCH} {REPO_URL}
    %cd {REPO_NAME}/benchmarking/notebooks

os.makedirs("../results", exist_ok=True)

Cloning into 'SLM-Software-Engineering'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 83 (delta 24), reused 41 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 203.03 KiB | 2.60 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/kaggle/working/SLM-Software-Engineering/benchmarking/notebooks


In [8]:
!pip uninstall -y -q torchao

In [2]:
!pip install -q psutil nvidia-ml-py
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-497c7341-7b3d-396c-36c7-a38a680ab0ea)
GPU 1: Tesla T4 (UUID: GPU-2723f3e5-df4b-9515-822f-e401ccce4ed4)


In [3]:
import json, time, sys, re, threading
from math import comb
import torch, psutil, pynvml
import pandas as pd
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

sys.path.append("..")
from utils.safe_exec import run_tests
from utils.extract import extract_code

# === SEULE LIGNE A CHANGER : "qwen", "phi", "gemma" ou "smollm" ===
MODEL_KEY = "qwen"
# ====================================================================

for f in ["../config.json", "../config_complement.json", "../prompts_securite.json", "../humaneval_subset.json"]:
    assert os.path.exists(f), f"{f} introuvable : ajoutez-le a la branche clonee (BRANCH) puis relancez."

config = json.load(open("../config.json"))
CC = json.load(open("../config_complement.json"))
SEC_PROMPTS = json.load(open("../prompts_securite.json", encoding="utf-8"))["prompts"]
PK, MON = CC["passk"], CC["monitor"]

MODEL_NAME = config["models"][MODEL_KEY]
GEN_PARAMS = config["generation_params"]
HE_CONFIG = config["humaneval"]
SYSTEM_PROMPT = config["no_think_models"].get(MODEL_KEY)
model_slug = MODEL_NAME.split("/")[-1].lower()
GPU_NAME = torch.cuda.get_device_name(0)
print(MODEL_NAME, "|", model_slug, "|", GPU_NAME, "| transformers", transformers.__version__)

Qwen/Qwen2.5-Coder-1.5B-Instruct | qwen2.5-coder-1.5b-instruct | Tesla T4 | transformers 5.20.2


In [4]:
# === CHANGE ONLY THIS LINE: "ft_bf16" or "base" ===
VARIANT = "ft_bf16"
# ==================================================
assert VARIANT in ("ft_bf16", "base")
BASE_SLUG = model_slug
model_slug = BASE_SLUG + ("-ft-run1-bf16" if VARIANT == "ft_bf16" else "-ctrl")
print("VARIANT:", VARIANT, "| results suffix:", model_slug)

VARIANT: ft_bf16 | results suffix: qwen2.5-coder-1.5b-instruct-ft-run1-bf16


In [5]:
pynvml.nvmlInit()
NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
print("NVML GPU 0 :", pynvml.nvmlDeviceGetName(NVML_HANDLE))

def gpu_power_w():
    return pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE) / 1000.0

class Monitor:
    def __init__(self, power_fn=gpu_power_w, interval=MON["sample_interval_s"]):
        self.power_fn, self.interval = power_fn, interval
    def __enter__(self):
        self.t, self.p, self.rss, self._stop = [], [], [], False
        self._proc = psutil.Process()
        self._th = threading.Thread(target=self._loop, daemon=True)
        self._t0 = time.time(); self._th.start()
        return self
    def _loop(self):
        while not self._stop:
            try:
                self.p.append(self.power_fn()); self.t.append(time.time())
                self.rss.append(self._proc.memory_info().rss)
            except Exception:
                pass
            time.sleep(self.interval)
    def __exit__(self, *a):
        self.duration = time.time() - self._t0
        self._stop = True; self._th.join()
    @property
    def avg_power_w(self):
        if len(self.p) >= 2:
            area = sum((self.p[i] + self.p[i+1]) / 2 * (self.t[i+1] - self.t[i]) for i in range(len(self.p) - 1))
            span = self.t[-1] - self.t[0]
            return area / span if span > 0 else self.p[0]
        return self.p[0] if self.p else float("nan")
    @property
    def peak_power_w(self):
        return max(self.p) if self.p else float("nan")
    @property
    def cpu_rss_peak_go(self):
        return max(self.rss) / 1e9 if self.rss else float("nan")

NVML GPU 0 : Tesla T4


In [9]:
import glob, zipfile, hashlib
from transformers import AutoModelForCausalLM, AutoTokenizer

assert config["precision"] == "bf16", "baseline precision changed: stop and ask the group"
torch.manual_seed(42)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)          # base tokenizer = same prompt format as the baseline
DTYPE = torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=DTYPE)
ADAPTER_META = None

if VARIANT == "ft_bf16":
    from peft import PeftModel
    cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if not cfgs:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        assert zips, "Add the adapter zip (adapter_run1.zip) as a Kaggle Input, then rerun"
        zipfile.ZipFile(zips[0]).extractall("/kaggle/working/adapter_run1")
        cfgs = glob.glob("/kaggle/working/adapter_run1/**/adapter_config.json", recursive=True)
    ADAPTER_DIR = os.path.dirname(cfgs[0])
    acfg = json.load(open(cfgs[0]))
    wfile = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
    ADAPTER_META = {"dir": ADAPTER_DIR, "base_model_name_or_path": acfg.get("base_model_name_or_path"),
                    "r": acfg.get("r"), "lora_alpha": acfg.get("lora_alpha"),
                    "target_modules": acfg.get("target_modules"),
                    "sha256": hashlib.sha256(open(wfile, "rb").read()).hexdigest()}
    print("adapter:", ADAPTER_META)
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    model = model.merge_and_unload()          # same speed profile as the base model
    assert not any("lora" in n.lower() for n, _ in model.named_parameters()), "LoRA layers still present"
model.eval()
print("model ready:", VARIANT, "| dtype:", next(model.parameters()).dtype)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter: {'dir': '/kaggle/input/datasets/minada43/adapter-run1-zip', 'base_model_name_or_path': 'unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit', 'r': 16, 'lora_alpha': 16, 'target_modules': ['down_proj', 'v_proj', 'k_proj', 'q_proj', 'gate_proj', 'o_proj', 'up_proj'], 'sha256': '02ec15117c3bb9e0e84b5a0a7eec98f0bf1d14f438d6fa592d56a3cbe90ab9f3'}
model ready: ft_bf16 | dtype: torch.bfloat16


In [10]:
torch.manual_seed(42)


def generate(user_prompt: str):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})

    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]

    # TTFT : temps pour produire 1 seul token
    torch.cuda.synchronize(); t0 = time.time()
    model.generate(**inputs, max_new_tokens=1, do_sample=False)
    torch.cuda.synchronize(); ttft = time.time() - t0

    # Generation complete
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                         do_sample=GEN_PARAMS["do_sample"])
    torch.cuda.synchronize(); latency = time.time() - t0

    new_tokens = out[0][n_in:]
    return {
        "response": tokenizer.decode(new_tokens, skip_special_tokens=True),
        "gpu": GPU_NAME,
        "ttft_s": round(ttft, 3),
        "latency_s": round(latency, 3),
        "tokens_per_sec": round(len(new_tokens) / latency, 2) if latency > 0 else None,
        "input_tokens": n_in,
        "output_tokens": len(new_tokens),
        "gpu_memory_go": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    }


# Échauffement + contrôle : le 1er appel est lent, et on vérifie que le modèle répond vraiment
test = generate("Write a Python function add(a, b) that returns a + b.")
print(test["response"][:300])
assert "def" in test["response"], "Réponse vide ou incohérente : essayez bf16 dans config.json, puis redémarrez la session."
print("Modele pret.")

```python
def add(a, b):
    return a + b
```

The `add` function takes two parameters, `a` and `b`, and returns their sum. This is a simple addition operation in Python.
Modele pret.


In [11]:
torch.manual_seed(PK["seed"])

def build_inputs(user_prompt):
    messages = []
    if SYSTEM_PROMPT:
        messages.append({"role": "system", "content": SYSTEM_PROMPT})
    messages.append({"role": "user", "content": user_prompt})
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(model.device)

def run_measured(user_prompt):
    # generation greedy (memes parametres que le benchmark de base) avec mesure de puissance
    inputs = build_inputs(user_prompt)
    n_in = inputs["input_ids"].shape[1]
    torch.cuda.synchronize()
    with Monitor() as mon:
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"],
                             do_sample=GEN_PARAMS["do_sample"])
        torch.cuda.synchronize()
        dur = time.time() - t0
    new = out[0][n_in:]
    avg_w = mon.avg_power_w
    return {
        "response": tokenizer.decode(new, skip_special_tokens=True),
        "latency_s": round(dur, 3), "output_tokens": len(new),
        "avg_power_w": round(avg_w, 1), "peak_power_w": round(mon.peak_power_w, 1),
        "energy_wh": round(avg_w * dur / 3600, 5),
        "cpu_rss_peak_go": round(mon.cpu_rss_peak_go, 2),
    }

# Echauffement + controle
test = run_measured("Write a Python function add(a, b) that returns a + b.")
assert "def" in test["response"], "Reponse incoherente : verifiez la version de transformers."
print("Modele pret. Test :", {k: v for k, v in test.items() if k != "response"})

# Consommation au repos (GPU charge mais inactif)
torch.cuda.synchronize()
with Monitor() as idle:
    time.sleep(MON["idle_seconds"])
IDLE_W = idle.avg_power_w
print(f"Puissance au repos : {IDLE_W:.1f} W")

Modele pret. Test : {'latency_s': 2.167, 'output_tokens': 48, 'avg_power_w': 58.7, 'peak_power_w': 62.9, 'energy_wh': 0.03533, 'cpu_rss_peak_go': 1.88}
Puissance au repos : 34.9 W


## Part A: 15 prompts

In [12]:
with open("../prompts.json") as f:
    prompts = json.load(f)["prompts"]

def norm(s):  # pour comparer sans tenir compte des espaces
    return re.sub(r"\s+", "", s)

results = []
for p in prompts:
    gen = generate(p["prompt"])
    response = gen.pop("response")

    status, error = None, None
    if p["task"] == "generation_code" and "test" in p:
        status, error = run_tests(extract_code(response), p["test"],
                                  timeout=HE_CONFIG["timeout_seconds"])
    elif p["task"] == "completion_code" and "expected" in p:
        status = "pass" if norm(p["expected"]) in norm(response) else "fail"
    # bug_detection, explication, refactoring, generation_tests : notes manuelles

    results.append({
        "id": p["id"], "task": p["task"], "prompt": p["prompt"], "response": response,
        **gen,
        "auto_status": status, "auto_error": error,
        # A remplir a la main (voir grille_notation.md et README)
        "bug_label": None, "note_qualite": None, "hallucination": None,
    })
    print(p["id"], status)

df_prompts = pd.DataFrame(results)
df_prompts.to_csv(f"../results/benchmark_{model_slug}.csv", index=False)
print(f"-> results/benchmark_{model_slug}.csv")
df_prompts[["id", "task", "ttft_s", "latency_s", "tokens_per_sec", "gpu_memory_go", "auto_status"]]

P1 pass
5
P2 pass
monde le Bonjour
P3 pass
P4 pass
P5 fail
P6 pass
P7 None
P8 None
P9 None
P10 None
P11 None
P12 None
P13 None
P14 None
P15 None
-> results/benchmark_qwen2.5-coder-1.5b-instruct-ft-run1-bf16.csv


,id,task,ttft_s,latency_s,tokens_per_sec,gpu_memory_go,auto_status
0,P1,generation_code,0.179,8.813,23.71,6.20,pass
1,P2,generation_code,0.091,8.583,23.77,6.20,pass
2,P3,generation_code,0.091,6.877,23.56,6.20,pass
3,P4,generation_code,0.097,6.810,23.94,6.20,pass
4,P5,completion_code,0.098,5.921,23.65,6.20,fail
5,P6,completion_code,0.150,5.324,24.04,6.20,pass
6,P7,bug_detection,0.095,5.162,24.02,6.19,None
7,P8,bug_detection,0.155,5.309,23.73,6.20,None
8,P9,bug_detection,0.098,6.185,23.60,6.20,None
9,P10,explication,0.158,1.932,22.77,6.19,None


## Part B: HumanEval (20 problems, greedy)

In [13]:
with open("../humaneval_subset.json") as f:
    subset = json.load(f)

humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
print(len(humaneval), "problemes")

HEADER = "from typing import *\nimport math\n\n"   # imports courants que le modele peut oublier

details = []
for ex in humaneval:
    gen = generate(HE_CONFIG["instruction"] + ex["prompt"])
    code_generated = extract_code(gen["response"])
    test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
    status, error = run_tests(HEADER + ex["prompt"] + "\n" + code_generated, test_code,
                              timeout=HE_CONFIG["timeout_seconds"])
    details.append({
        "task_id": ex["task_id"], "status": status, "error": error,
        "ttft_s": gen["ttft_s"], "latency_s": gen["latency_s"],
        "tokens_per_sec": gen["tokens_per_sec"],
        "output_tokens": gen["output_tokens"], "gpu_memory_go": gen["gpu_memory_go"],
        "response": gen["response"],
    })
    print(ex["task_id"], status)

n_pass = sum(d["status"] == "pass" for d in details)
summary = {
    "model": MODEL_NAME,
    "gpu": GPU_NAME,
    "precision": config["precision"],
    "dataset": "HumanEval (sous-ensemble des 20 premiers problemes)",
    "n_problems": len(details),
    "n_pass": n_pass,
    "pass@1": round(100 * n_pass / len(details), 1),
    "details": details,
}
with open(f"../results/humaneval_{model_slug}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f"pass@1 = {summary['pass@1']}% ({n_pass}/{len(details)}) -> results/humaneval_{model_slug}.json")

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Filter:   0%|          | 0/164 [00:00<?, ? examples/s]

20 problemes
HumanEval/0 pass
HumanEval/1 pass
HumanEval/2 pass
HumanEval/3 pass
HumanEval/4 pass
HumanEval/5 pass
HumanEval/6 fail
HumanEval/7 pass
HumanEval/8 pass
HumanEval/9 pass
HumanEval/10 pass
HumanEval/11 pass
HumanEval/12 pass
HumanEval/13 pass
HumanEval/14 pass
HumanEval/15 pass
HumanEval/16 pass
HumanEval/17 pass
HumanEval/18 pass
HumanEval/19 fail
pass@1 = 90.0% (18/20) -> results/humaneval_qwen2.5-coder-1.5b-instruct-ft-run1-bf16.json


## Energy + 9 security prompts

In [14]:
base_prompts = json.load(open("../prompts.json"))["prompts"]
rows, vuln_rows = [], []

for group, plist in [("base", base_prompts), ("securite", SEC_PROMPTS)]:
    for p in plist:
        r = run_measured(p["prompt"])
        resp = r.pop("response")
        r["energy_net_wh"] = round(max(0.0, (r["avg_power_w"] - IDLE_W) * r["latency_s"] / 3600), 5)
        r["tokens_per_sec"] = round(r["output_tokens"] / r["latency_s"], 2) if r["latency_s"] > 0 else None
        rows.append({"id": p["id"], "task": p["task"], **r})
        if group == "securite":
            kw = [k.lower() for k in p.get("keywords", [])]
            vuln_rows.append({
                "id": p["id"], "prompt": p["prompt"], "response": resp,
                "has_vuln": p["has_vuln"], "expected_vuln": p["expected_vuln"],
                "mot_cle_trouve": any(k in resp.lower() for k in kw) if kw else None,
                "vuln_label": None,    # a remplir a la main (partie 3)
            })
        print(p["id"], f'{r["latency_s"]}s', f'{r["energy_wh"]} Wh')

df_energy = pd.DataFrame(rows)
df_energy.to_csv(f"../results/energie_{model_slug}.csv", index=False)
pd.DataFrame(vuln_rows).to_csv(f"../results/vulnerabilites_{model_slug}.csv", index=False)
print(f"-> results/energie_{model_slug}.csv, results/vulnerabilites_{model_slug}.csv")
df_energy[["id", "latency_s", "avg_power_w", "energy_wh", "energy_net_wh", "cpu_rss_peak_go"]]

P1 8.401s 0.15722 Wh
P2 8.243s 0.15358 Wh
P3 6.637s 0.1236 Wh
P4 6.669s 0.12439 Wh
P5 5.618s 0.10451 Wh
P6 5.277s 0.09753 Wh
P7 5.104s 0.09436 Wh
P8 5.146s 0.09547 Wh
P9 5.816s 0.10846 Wh
P10 1.916s 0.03529 Wh
P11 9.524s 0.17749 Wh
P12 3.865s 0.07179 Wh
P13 5.766s 0.10531 Wh
P14 8.1s 0.15176 Wh
P15 5.307s 0.09958 Wh
V1 12.816s 0.23964 Wh
V2 2.957s 0.05526 Wh
V3 10.163s 0.18816 Wh
V4 4.889s 0.09028 Wh
V5 4.252s 0.07866 Wh
V6 4.308s 0.07971 Wh
V7 1.23s 0.02224 Wh
V8 9.803s 0.18254 Wh
V9 5.757s 0.10626 Wh
-> results/energie_qwen2.5-coder-1.5b-instruct-ft-run1-bf16.csv, results/vulnerabilites_qwen2.5-coder-1.5b-instruct-ft-run1-bf16.csv


,id,latency_s,avg_power_w,energy_wh,energy_net_wh,cpu_rss_peak_go
0,P1,8.401,67.4,0.15722,0.07589,1.89
1,P2,8.243,67.1,0.15358,0.07378,1.89
2,P3,6.637,67.0,0.12360,0.05922,1.89
3,P4,6.669,67.1,0.12439,0.05969,1.89
4,P5,5.618,67.0,0.10451,0.05013,1.89
5,P6,5.277,66.5,0.09753,0.04635,1.89
6,P7,5.104,66.6,0.09436,0.04497,1.89
7,P8,5.146,66.8,0.09547,0.04563,1.89
8,P9,5.816,67.1,0.10846,0.05205,1.89
9,P10,1.916,66.3,0.03529,0.01672,1.89


## pass@k (sampling)

In [15]:
def pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

def sample_n(user_prompt, n):
    inputs = build_inputs(user_prompt)
    n_in = inputs["input_ids"].shape[1]
    out = model.generate(**inputs, max_new_tokens=GEN_PARAMS["max_new_tokens"], do_sample=True,
                         temperature=PK["temperature"], top_p=PK["top_p"], top_k=PK["top_k"],
                         num_return_sequences=n)
    return [tokenizer.decode(o[n_in:], skip_special_tokens=True) for o in out]

subset = json.load(open("../humaneval_subset.json"))
humaneval = load_dataset(HE_CONFIG["dataset"], split="test")
humaneval = humaneval.filter(lambda ex: ex["task_id"] in subset["task_ids"])
HEADER = "from typing import *\nimport math\n\n"
N = PK["n_samples"]

per_problem = []
for i, ex in enumerate(humaneval):
    torch.manual_seed(PK["seed"] + i)
    t0 = time.time()
    samples = sample_n(HE_CONFIG["instruction"] + ex["prompt"], N)
    test_code = ex["test"] + f"\ncheck({ex['entry_point']})"
    c = 0
    for s in samples:
        st, _ = run_tests(HEADER + ex["prompt"] + "\n" + extract_code(s), test_code,
                          timeout=HE_CONFIG["timeout_seconds"])
        c += (st == "pass")
    per_problem.append({"task_id": ex["task_id"], "n": N, "c": c})
    print(ex["task_id"], f"{c}/{N} corrects", f"({time.time() - t0:.0f}s)")

pass_k = {str(k): round(100 * sum(pass_at_k(p["n"], p["c"], k) for p in per_problem) / len(per_problem), 1)
          for k in PK["k_values"]}
summary = {"model": MODEL_NAME, "gpu": GPU_NAME, "precision": config["precision"],
           "n_problems": len(per_problem), "n_samples": N, "temperature": PK["temperature"],
           "top_p": PK["top_p"], "top_k": PK["top_k"], "seed": PK["seed"],
           "pass_at_k": pass_k, "per_problem": per_problem}
with open(f"../results/passk_{model_slug}.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("pass@k (%) :", pass_k, f"-> results/passk_{model_slug}.json")

HumanEval/0 9/10 corrects (22s)
HumanEval/1 9/10 corrects (25s)
HumanEval/2 10/10 corrects (13s)
HumanEval/3 10/10 corrects (20s)
HumanEval/4 10/10 corrects (24s)
HumanEval/5 5/10 corrects (20s)
HumanEval/6 3/10 corrects (26s)
HumanEval/7 10/10 corrects (14s)
HumanEval/8 10/10 corrects (21s)
HumanEval/9 10/10 corrects (19s)
HumanEval/10 6/10 corrects (28s)
HumanEval/11 8/10 corrects (21s)
HumanEval/12 10/10 corrects (23s)
HumanEval/13 10/10 corrects (13s)
HumanEval/14 10/10 corrects (11s)
HumanEval/15 9/10 corrects (12s)
HumanEval/16 10/10 corrects (11s)
HumanEval/17 8/10 corrects (32s)
HumanEval/18 8/10 corrects (20s)
HumanEval/19 5/10 corrects (32s)
pass@k (%) : {'1': 85.0, '5': 99.5, '10': 100.0} -> results/passk_qwen2.5-coder-1.5b-instruct-ft-run1-bf16.json


In [16]:
import platform, datetime
from importlib.metadata import version
meta = {"variant": VARIANT, "slug": model_slug, "model": MODEL_NAME, "gpu": GPU_NAME,
        "precision": config["precision"], "adapter": ADAPTER_META,
        "generation_params": GEN_PARAMS, "passk": PK,
        "versions": {k: version(k) for k in ["torch","transformers","peft","accelerate","datasets","pandas","psutil"]},
        "python": platform.python_version(), "date_utc": datetime.datetime.utcnow().isoformat()}
json.dump(meta, open(f"../results/run_meta_{model_slug}.json", "w"), indent=2)
print(json.dumps(meta, indent=2))

{
  "variant": "ft_bf16",
  "slug": "qwen2.5-coder-1.5b-instruct-ft-run1-bf16",
  "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
  "gpu": "Tesla T4",
  "precision": "bf16",
  "adapter": {
    "dir": "/kaggle/input/datasets/minada43/adapter-run1-zip",
    "base_model_name_or_path": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "r": 16,
    "lora_alpha": 16,
    "target_modules": [
      "down_proj",
      "v_proj",
      "k_proj",
      "q_proj",
      "gate_proj",
      "o_proj",
      "up_proj"
    ],
    "sha256": "02ec15117c3bb9e0e84b5a0a7eec98f0bf1d14f438d6fa592d56a3cbe90ab9f3"
  },
  "generation_params": {
    "max_new_tokens": 512,
    "do_sample": false
  },
  "passk": {
    "n_samples": 10,
    "k_values": [
      1,
      5,
      10
    ],
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 50,
    "seed": 42
  },
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "peft": "0.21.2",
    "accelerate": "1.15.0",
    "datasets": "4.8.5",
    "p

/tmp/ipykernel_49/4089356702.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "python": platform.python_version(), "date_utc": datetime.datetime.utcnow().isoformat()}


In [17]:
import zipfile
names = [f"results/{p}_{model_slug}.{e}" for p, e in [("benchmark","csv"),("humaneval","json"),("energie","csv"),
         ("vulnerabilites","csv"),("passk","json"),("run_meta","json")]]
out = f"/kaggle/working/eval_{model_slug}.zip"
with zipfile.ZipFile(out, "w") as z:
    for n in names:
        z.write("../" + n, arcname=n)
print("Created:", out)

Created: /kaggle/working/eval_qwen2.5-coder-1.5b-instruct-ft-run1-bf16.zip
